# 00 Bootstrap — VIST-NRF on Colab
Mounts Google Drive, sets `VIST_NRF_DRIVE/DATA/RESULTS`, clones the repo via Git using `GITHUB_TOKEN` from Colab Secrets, and installs the package in editable mode.

In [ ]:
import os, stat, subprocess
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
os.environ['VIST_NRF_DRIVE'] = '/content/drive/MyDrive/VIST-NRF'
os.environ['VIST_NRF_DATA'] = str(Path(os.environ['VIST_NRF_DRIVE']) / 'data')
os.environ['VIST_NRF_RESULTS'] = str(Path(os.environ['VIST_NRF_DRIVE']) / 'results')
for key in ('VIST_NRF_DATA', 'VIST_NRF_RESULTS'):
    Path(os.environ[key]).mkdir(parents=True, exist_ok=True)

repo_url = 'https://github.com/<OWNER>/VIST-NRF.git'  # <-- set OWNER
repo_dir = '/content/VIST-NRF'
token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('Add GITHUB_TOKEN to Colab Secrets.')

askpass = Path('/tmp/vist_nrf_askpass.sh')
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo x-access-token ;;
  *Password*) printf "%s" "$GITHUB_TOKEN" ;;
esac\n')
askpass.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)
env = os.environ.copy()
env.update(GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT='0', GITHUB_TOKEN=token)
try:
    if not Path(repo_dir).exists():
        subprocess.run(['git', 'clone', repo_url, repo_dir], check=True, env=env)
    else:
        subprocess.run(['git', '-C', repo_dir, 'pull', '--ff-only'], check=True, env=env)
finally:
    askpass.unlink(missing_ok=True)
    env.pop('GITHUB_TOKEN', None)
    del token

subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', repo_dir], check=True)
import torch, vist_nrf
commit = subprocess.check_output(['git', '-C', repo_dir, 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| commit:', commit)